# Neural Networks for Bank Customer Churn

This notebook extends Midterm 1 by fitting a Multi-Layer Perceptron (MLP) to the Bank Customer Churn dataset using PyTorch. The goal is to examine whether a neural network can improve churn prediction relative to the probabilistic models studied previously, particularly by capturing non-linear effects and interactions.

The analysis preserves the same target variable, `exited`, and the same asymmetric business setting from Midterm 1. As before, `Complain` is excluded due to data leakage. Because the network is trained with a class-weighted loss, the cost-based threshold of approximately **0.09** is used as a theoretical reference, while the final operating threshold is determined empirically.

The notebook is organised as follows:

1. Data loading and preprocessing
2. Dataset preparation
3. Model architecture
4. Training procedure
5. Baseline MLP
6. Hyperparameter tuning
7. Threshold analysis and cost evaluation
8. SHAP-based interpretability
9. Comparison with Midterm 1 models




In [ ]:
# Install required packages (uncomment if running for the first time)
# %pip install torch matplotlib seaborn scikit-learn pandas shap -q

In [ ]:
# Imports, reproducibility seed, and device selection (CPU/GPU).
import random
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)
import shap

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Data Loading and Preprocessing

To ensure comparability with Midterm 1, we keep the same preprocessing pipeline. Identifier variables are removed, `Complain` is excluded due to leakage, and categorical predictors are one-hot encoded. Feature standardisation is performed later using only the training split, so that no information from validation or test data leaks into model fitting.

In [ ]:
# Load the dataset.
# Adjust the path/filename
df = pd.read_csv("Customer-Churn-Records.csv")

print(f"Shape: {df.shape}")
df.head(3)

In [ ]:
# ── 1. Standardise column names ──────────────────────────────────────────────
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# ── 2. Drop identifiers and data-leakage variable ────────────────────────────
drop_cols = ["rownumber", "customerid", "surname", "complain"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# ── 3. One-hot encode categoricals ───────────────────────────────────────────
# We keep drop_first=True for consistency with Midterm 1 and to avoid redundant dummy columns.
cat_cols = ["geography", "gender", "card_type", "has_cr_card", "is_active_member"]
cat_cols = [c for c in cat_cols if c in df.columns]

df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# ── 4. Separate features and target ──────────────────────────────────────────
target = "exited"
feature_names = [c for c in df.columns if c != target]

X_raw = df[feature_names].values.astype(np.float32)
y_raw = df[target].values.astype(np.float32)

print(f"Features: {len(feature_names)}")
print(feature_names)
print(f"\nClass distribution:")
print(f"  No churn (0): {(y_raw == 0).sum()}  ({(y_raw == 0).mean():.1%})")
print(f"  Churn   (1): {(y_raw == 1).sum()}  ({(y_raw == 1).mean():.1%})")

In [ ]:
# Visualise class imbalance — same imbalance ratio as in Midterm 1.
fig, ax = plt.subplots(figsize=(5, 3))
counts = pd.Series(y_raw).value_counts().sort_index()
ax.bar(["No churn (0)", "Churn (1)"], counts.values,
       color=["steelblue", "#FFA07A"])
ax.set_ylabel("Count")
ax.set_title("Target class distribution")
for i, v in enumerate(counts.values):
    ax.text(i, v + 50, f"{v} ({v/len(y_raw):.1%})", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## 2. Dataset Preparation

As in Midterm 1, we retain an 80/20 train-test split, but we now further divide the training portion into a training and validation set. The validation set is used for early stopping, model selection, and later threshold tuning, while the test set is kept untouched for final evaluation.

To address class imbalance, the neural network is trained with a class-weighted loss rather than with external resampling. In PyTorch, this is implemented through the `pos_weight` argument in `BCEWithLogitsLoss`, which increases the penalty for misclassifying churners during training. This does not replicate SMOTE or oversampling exactly, but it serves the same practical goal of giving more importance to the minority class.

The positive-class weight is defined as

$$
\text{pos\_weight} = \frac{N_{\text{no churn}}}{N_{\text{churn}}}
$$

so that errors on churners receive a larger penalty during training.

In [ ]:
def build_churn_loaders(
    X, y,
    batch_size=64,
    test_size=0.2,
    val_size=0.2,
    seed=42,
):
    """Build train/val/test DataLoaders for binary churn classification.

    Args:
        X: feature array (n_samples, n_features).
        y: binary label array (n_samples,).
        batch_size: mini-batch size.
        test_size: fraction reserved for final test evaluation.
        val_size: fraction of training data used for validation.
        seed: random state for reproducible splits.

    Returns:
        train_loader, val_loader, test_loader, scaler, pos_weight

    Notes:
        - stratify=y ensures both splits preserve the 80/20 class ratio.
        - StandardScaler is fit only on training data to avoid data leakage.
        - pos_weight = n_neg / n_pos is used inside BCEWithLogitsLoss to
          compensate for class imbalance without resampling.
    """
    # 80 / 20 train-test split (stratified)
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=seed
    )

    # Split training into train + validation (stratified)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=val_size, stratify=y_train_full, random_state=seed
    )

    # Standardise: fit on train, transform all splits
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    # Positive class weight to handle imbalance
    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32)

    def _make_ds(Xn, yn):
        return TensorDataset(
            torch.tensor(Xn, dtype=torch.float32),
            torch.tensor(yn, dtype=torch.float32).unsqueeze(1),  # (n,1) for BCEWithLogitsLoss
        )

    train_loader = DataLoader(_make_ds(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(_make_ds(X_val,   y_val),   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(_make_ds(X_test,  y_test),  batch_size=batch_size, shuffle=False)

    print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
    print(f"pos_weight (n_neg/n_pos): {pos_weight.item():.2f}")

    return train_loader, val_loader, test_loader, scaler, pos_weight, X_test, y_test


train_loader, val_loader, test_loader, scaler, pos_weight, X_test_np, y_test_np = build_churn_loaders(
    X_raw, y_raw, batch_size=64
)

## 3. Model Architecture

We use a feed-forward Multi-Layer Perceptron (MLP) with two hidden layers for binary classification. The network maps the input feature vector into a sequence of non-linear transformations, allowing it to capture interactions and non-linear effects that could not be represented by the probabilistic models considered in Midterm 1.

Its architecture is intentionally kept simple:

```text
Input (n_features)
  → Linear → ReLU → Dropout
  → Linear → ReLU
  → Linear → 1 logit
```
The hidden-layer sizes are chosen to keep the network deliberately small and appropriate for a tabular dataset of moderate size. This provides a reasonable baseline specification without introducing unnecessary complexity, while alternative configurations are examined later in the tuning stage.

The final layer produces a **single logit**, which is converted into a probability through the sigmoid function at prediction time. During training, we use `BCEWithLogitsLoss`, which combines the sigmoid transformation and binary cross-entropy loss in a numerically stable way.

This architecture can be viewed as a flexible extension of logistic regression: while logistic regression is linear in the inputs, the hidden layers and ReLU activations allow the MLP to learn more complex decision boundaries.

In [ ]:
class ChurnMLP(nn.Module):
    def __init__(self, input_dim, hidden1=32, hidden2=16, dropout=0.2):
        """MLP binary classifier for customer churn.

        Args:
            input_dim: number of input features after one-hot encoding.
            hidden1: neurons in the first hidden layer.
            hidden2: neurons in the second hidden layer.
            dropout: dropout probability applied after the first ReLU.
                     Prevents co-adaptation of neurons (regularisation).

        Output:
            Raw logit (shape [batch, 1]). Apply sigmoid for probabilities.
        """
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),          # randomly zeroes neurons during training
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, 1),        # single logit for binary classification
        )

    def forward(self, x):
        return self.net(x)


input_dim = X_raw.shape[1]
demo_model = ChurnMLP(input_dim=input_dim)
print(demo_model)
total_params = sum(p.numel() for p in demo_model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

## 4. Training Utilities

The following functions implement the training and evaluation workflow for binary classification. The network is trained with a weighted binary cross-entropy loss, predicted probabilities are obtained by applying the sigmoid function to the output logits, and performance is monitored on the validation set throughout training.

To avoid overfitting, early stopping is applied based on validation loss, with the model parameters restored from the best-performing epoch. Additional utilities are included to generate predicted probabilities, visualise learning curves, and evaluate performance across alternative decision thresholds under the asymmetric cost setting defined earlier.


In [ ]:
def train_one_epoch_binary(model, loader, criterion, optimizer, device):
    """Train for one epoch (binary classification).

    Returns:
        (avg_loss, accuracy) over the epoch.
    """
    model.train()   # enables dropout
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)                         # raw scores, shape (batch, 1)
        loss = criterion(logits, y)               # BCEWithLogitsLoss applies sigmoid internally
        loss.backward()                           # compute gradients via backpropagation
        optimizer.step()                          # update weights

        total_loss += loss.item() * x.size(0)
        preds = (torch.sigmoid(logits) > 0.5).float()   # threshold at 0.5 for training accuracy
        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total


def evaluate_binary(model, loader, criterion, device):
    """Evaluate binary classifier without gradient tracking.

    Returns:
        (avg_loss, accuracy)
    """
    model.eval()   # disables dropout
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss / total, correct / total


def predict_proba_binary(model, loader, device):
    """Return predicted probabilities and true labels as NumPy arrays.

    Returns:
        y_true (n,), y_prob (n,)  — both as float32 arrays.
    """
    model.eval()
    y_true_list, y_prob_list = [], []

    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(device))
            probs = torch.sigmoid(logits).cpu().squeeze(1)   # (batch,)
            y_prob_list.extend(probs.numpy().tolist())
            y_true_list.extend(y.squeeze(1).numpy().tolist())

    return np.array(y_true_list), np.array(y_prob_list)


def fit_binary(
    model, train_loader, val_loader, criterion, optimizer,
    epochs, device, patience=10
):
    """Full training loop with early stopping.

    Args:
        patience: stop if val_loss does not improve for this many epochs.
                  Restores weights from the best epoch.

    Returns:
        history dict with train/val loss and accuracy per epoch.
    """
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch_binary(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate_binary(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        dt = time.time() - t0
        print(
            f"Epoch {epoch:03d}/{epochs} | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f} | {dt:.1f}s"
        )

        # ── Early stopping ──────────────────────────────────────────────────
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch} (no improvement for {patience} epochs).")
                break

    # Restore best weights
    model.load_state_dict(best_state)
    print(f"Best val_loss: {best_val_loss:.4f}")
    return history


def plot_learning_curves(history, title_prefix="Model"):
    """Plot loss and accuracy learning curves from the history dictionary."""
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history["train_loss"], marker="o", label="Train")
    axes[0].plot(epochs, history["val_loss"],   marker="o", label="Validation")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title(f"{title_prefix} — Loss curves")
    axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], marker="o", label="Train")
    axes[1].plot(epochs, history["val_acc"],   marker="o", label="Validation")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
    axes[1].set_title(f"{title_prefix} — Accuracy curves")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def evaluate_at_threshold(y_true, y_prob, threshold, cost_fn=100, cost_fp=10):
    """Compute all evaluation metrics at a given classification threshold.

    Args:
        y_true: true binary labels.
        y_prob: predicted probabilities (from sigmoid).
        threshold: decision boundary in [0, 1].
        cost_fn: business cost of a False Negative (missed churner).
        cost_fp: business cost of a False Positive (wrongly flagged loyal customer).

    Note on costs: in Midterm 1, FP=10 (intervention cost) and FN=100 (lost customer).
    The theoretical threshold is cost_fp / (cost_fp + cost_fn) = 10/110 ≈ 0.09.
    """
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    sensitivity  = tp / (tp + fn) if (tp + fn) > 0 else 0.0   # recall / true positive rate
    specificity  = tn / (tn + fp) if (tn + fp) > 0 else 0.0   # true negative rate
    precision    = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    accuracy     = (tp + tn) / (tp + tn + fp + fn)
    f1           = 2 * precision * sensitivity / (precision + sensitivity + 1e-9)
    auc          = roc_auc_score(y_true, y_prob)
    expected_cost = (cost_fn * fn + cost_fp * fp) / len(y_true)  # average cost per customer

    return {
        "threshold":      threshold,
        "AUC":            round(auc, 3),
        "Accuracy":       round(accuracy, 3),
        "Sensitivity":    round(sensitivity, 3),
        "Specificity":    round(specificity, 3),
        "Precision":      round(precision, 3),
        "F1_Score":       round(f1, 3),
        "Expected_Cost":  round(expected_cost, 3),
    }

print("Training utilities defined.")

## 5. Baseline MLP

Before tuning, we fit a baseline MLP using a simple and reasonable default configuration. The purpose of this model is not to maximise performance, but to provide a stable reference point against which later tuning results can be evaluated.

We first inspect the learning curves to check whether the model trains in a stable way and whether there are visible signs of underfitting or overfitting. We then evaluate the baseline model in terms of discrimination (ROC-AUC) and threshold-dependent classification performance under the asymmetric cost setting introduced earlier.

In [ ]:
# Baseline configuration 
baseline_cfg = {
    "hidden1":  64,
    "hidden2":  32,
    "dropout":  0.2,
    "lr":       1e-3,
    "epochs":   100,     # early stopping will likely trigger before this
    "patience": 5,
    "batch_size": 64,
}

baseline_model = ChurnMLP(
    input_dim=input_dim,
    hidden1=baseline_cfg["hidden1"],
    hidden2=baseline_cfg["hidden2"],
    dropout=baseline_cfg["dropout"],
).to(device)

# BCEWithLogitsLoss: numerically stable binary cross-entropy that fuses sigmoid + BCE.
# pos_weight upweights churners in the loss so the model pays more attention to them.
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

# Adam: adaptive learning rate optimizer, the standard choice for MLPs.
optimizer = torch.optim.Adam(baseline_model.parameters(), lr=baseline_cfg["lr"])

baseline_history = fit_binary(
    baseline_model, train_loader, val_loader, criterion, optimizer,
    epochs=baseline_cfg["epochs"],
    device=device,
    patience=baseline_cfg["patience"],
)

 The selected configuration, (64, 32) with dropout 0.3 and learning rate 0.001, achieved the best validation performance within the grid, suggesting that a moderately sized network with slightly stronger regularisation generalises best on the validation set. Its learning curves remain broadly stable, although they still indicate mild overfitting, as training loss keeps decreasing while validation performance largely plateaus. However, this validation advantage does not translate into a better test result: the baseline MLP attains a test AUC of 0.8636, whereas the tuned model reaches 0.8614, a negligible difference of -0.0023. Overall, tuning does not produce a meaningful improvement over the baseline, which suggests that the simpler specification already captures most of the predictive signal in this dataset.

In [ ]:
# Learning curves: check for overfitting (train >> val) or underfitting (both high loss).
plot_learning_curves(baseline_history, title_prefix="Baseline MLP")

The learning curves suggest that the baseline network learns useful structure during the first epochs, but they also show slight overfitting afterwards. Training loss continues to decrease steadily, whereas validation loss reaches a plateau. A similar pattern appears in the accuracy curves, training accuracy keeps improving, while validation accuracy stabilises and becomes more erratic. This indicates that most of the useful learning occurs early in training, which supports the use of early stopping.

In [ ]:
# Evaluate baseline on the test set at the default threshold (0.5).
y_true_base, y_prob_base = predict_proba_binary(baseline_model, test_loader, device)

metrics_base_05  = evaluate_at_threshold(y_true_base, y_prob_base, threshold=0.5)
metrics_base_009 = evaluate_at_threshold(y_true_base, y_prob_base, threshold=0.09)

results_base = pd.DataFrame([metrics_base_05, metrics_base_009])
results_base.insert(0, "Strategy", ["Baseline (Default 0.5)", "Baseline (Cost-Sens 0.09)"])

In [ ]:
# ROC curve for the baseline model.
fpr, tpr, _ = roc_curve(y_true_base, y_prob_base)
auc_base = roc_auc_score(y_true_base, y_prob_base)

plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"Baseline MLP (AUC = {auc_base:.3f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Baseline MLP")
plt.legend()
plt.tight_layout()
plt.show()

In terms of ranking performance, the baseline MLP already shows strong discriminatory ability, with a test ROC-AUC of 0.860. This suggests that even a simple neural-network specification is capable of separating churners from non-churners reasonably well before any hyperparameter tuning.

## 6. Hyperparameter Tuning
We perform a small grid to assess whether modest changes in network capacity and regularisation improve performance relative to the baseline specification. The tuning focuses on three hyperparameters with the largest practical impact in this setting: hidden-layer size, dropout rate, and learning rate.

Candidate models are ranked by validation AUC, which provides a threshold-independent measure of ranking performance and is therefore appropriate for model selection at this stage. This keeps the tuning process separate from the later threshold-dependent evaluation.


In [ ]:
def run_churn_experiment(cfg, train_loader, val_loader, input_dim, device, pos_weight, epochs=80, patience=10):
    """Train one MLP configuration and return its best validation AUC.

    Args:
        cfg: dict with hidden1, hidden2, dropout, lr, batch_size.
        epochs: maximum training epochs (early stopping may cut this short).
        patience: early stopping patience.

    Returns:
        dict with config, model, best val_auc, and training history.
    """
    model = ChurnMLP(
        input_dim=input_dim,
        hidden1=cfg["hidden1"],
        hidden2=cfg["hidden2"],
        dropout=cfg["dropout"],
    ).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])

    history = fit_binary(
        model, train_loader, val_loader, criterion, optimizer,
        epochs=epochs, device=device, patience=patience
    )

    # Compute AUC on validation set (threshold-independent ranking criterion)
    y_true_v, y_prob_v = predict_proba_binary(model, val_loader, device)
    val_auc = roc_auc_score(y_true_v, y_prob_v)

    return {"config": cfg, "model": model, "val_auc": val_auc, "history": history}


# Grid of candidate configurations
grid = [
    {"hidden1": 32,  "hidden2": 16, "dropout": 0.2, "lr": 1e-3},
    {"hidden1": 32,  "hidden2": 16, "dropout": 0.3, "lr": 1e-3},
    {"hidden1": 64,  "hidden2": 32, "dropout": 0.2, "lr": 1e-3},
    {"hidden1": 64,  "hidden2": 32, "dropout": 0.3, "lr": 1e-3},
    {"hidden1": 128, "hidden2": 64, "dropout": 0.2, "lr": 1e-3},
    {"hidden1": 128, "hidden2": 64, "dropout": 0.3, "lr": 1e-3},
    {"hidden1": 128, "hidden2": 64, "dropout": 0.2, "lr": 5e-4},
    {"hidden1": 256, "hidden2": 128, "dropout": 0.3, "lr": 1e-3},
]

tuning_results = []
for i, cfg in enumerate(grid, start=1):
    print("=" * 70)
    print(f"Run {i}/{len(grid)} | {cfg}")
    result = run_churn_experiment(cfg, train_loader, val_loader, input_dim, device, pos_weight)
    tuning_results.append(result)

tuning_results = sorted(tuning_results, key=lambda r: r["val_auc"], reverse=True)

print("\n── Ranked tuning results (by val AUC) ──")
for rank, r in enumerate(tuning_results, start=1):
    cfg = r["config"]
    print(
        f"{rank}. val_auc={r['val_auc']:.4f} | "
        f"hidden=({cfg['hidden1']},{cfg['hidden2']}), "
        f"dropout={cfg['dropout']}, lr={cfg['lr']}"
    )

best_cfg = tuning_results[0]["config"]
best_model_tuned = tuning_results[0]["model"]
print(f"\nBest config: {best_cfg}")

In [ ]:
# Learning curves for the best configuration.
plot_learning_curves(tuning_results[0]["history"], title_prefix="Tuned MLP (best config)")

In [ ]:
from sklearn.metrics import roc_auc_score

# Baseline test AUC
y_true_base_check, y_prob_base_check = predict_proba_binary(baseline_model, test_loader, device)
baseline_test_auc = roc_auc_score(y_true_base_check, y_prob_base_check)

# Tuned test AUC
y_true_tuned_check, y_prob_tuned_check = predict_proba_binary(best_model_tuned, test_loader, device)
tuned_test_auc = roc_auc_score(y_true_tuned_check, y_prob_tuned_check)

print(f"Baseline test AUC: {baseline_test_auc:.4f}")
print(f"Tuned test AUC:    {tuned_test_auc:.4f}")
print(f"Delta (tuned - baseline): {tuned_test_auc - baseline_test_auc:+.4f}")


The selected configuration, (64, 32) with dropout 0.3 and learning rate 0.001, achieved the best validation performance within the grid, suggesting that a moderately sized network with slightly stronger regularisation generalises best on the validation set. Its learning curves remain broadly stable, although they still indicate mild overfitting, as training loss keeps decreasing while validation performance largely plateaus. However, this validation advantage does not translate into a better test result: the baseline MLP attains a test AUC of 0.8636, whereas the tuned model reaches 0.8614, a negligible difference of -0.0023. Overall, tuning does not produce a meaningful improvement over the baseline, which suggests that the simpler specification already captures most of the predictive signal in this dataset.

## 7. Threshold Analysis and Cost Evaluation

To complement the threshold-independent AUC analysis, we examine how the tuned MLP behaves under different decision thresholds. Since the churn problem is defined under asymmetric misclassification costs, performance is also assessed in terms of **expected business cost**:

$$
\text{Expected Cost} = \frac{100 \cdot \text{FN} + 10 \cdot \text{FP}}{N}
$$

Under perfectly calibrated probabilities, this cost structure suggests a theoretical reference threshold of

$$
\tau^* = \frac{\text{cost}*{FP}}{\text{cost}*{FP} + \text{cost}_{FN}} = \frac{10}{110} \approx 0.09.
$$

Because the neural network is trained with a class-weighted loss, this value is treated as a reference rather than as an exact optimum. The threshold sweep therefore serves to illustrate how expected cost, sensitivity, and specificity vary across alternative operating points.

In [ ]:
# Select threshold on validation set, then evaluate on test set.

# 1) Validation probabilities for threshold selection
y_true_val, y_prob_val = predict_proba_binary(best_model_tuned, val_loader, device)

# Sweep thresholds from 0.01 to 0.99 on VALIDATION
thresholds = np.arange(0.01, 1.00, 0.01)
sweep_results = [evaluate_at_threshold(y_true_val, y_prob_val, t) for t in thresholds]
sweep_df = pd.DataFrame(sweep_results)

optimal_idx  = sweep_df["Expected_Cost"].idxmin()
optimal_thr  = sweep_df.loc[optimal_idx, "threshold"]
optimal_cost = sweep_df.loc[optimal_idx, "Expected_Cost"]
print(f"Validation-optimal threshold: {optimal_thr:.2f}  (Validation Expected Cost = {optimal_cost:.3f})")

y_true_best, y_prob_best = predict_proba_binary(best_model_tuned, test_loader, device)

The validation sweep identifies 0.25 as the cost-minimising threshold, which is clearly above the theoretical reference value. This suggests that, in practice, the very low theoretical threshold is too aggressive for this model, generating too many false positives. 

In [ ]:
# Plot validation-based threshold selection diagnostics.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: expected cost curve on validation
axes[0].plot(sweep_df["threshold"], sweep_df["Expected_Cost"], color="darkblue")
axes[0].axvline(x=optimal_thr, color="red",    linestyle="--", label=f"Validation opt. ({optimal_thr:.2f})")
axes[0].axvline(x=0.09,        color="orange", linestyle="--", label="Theoretical (0.09)")
axes[0].axvline(x=0.50,        color="gray",   linestyle=":",  label="Default (0.50)")
axes[0].set_xlabel("Threshold"); axes[0].set_ylabel("Expected Cost per customer")
axes[0].set_title("Validation: Expected Cost vs Threshold")
axes[0].legend()

# Right: sensitivity/specificity trade-off on validation
axes[1].plot(sweep_df["threshold"], sweep_df["Sensitivity"], label="Sensitivity", color="tomato")
axes[1].plot(sweep_df["threshold"], sweep_df["Specificity"], label="Specificity", color="steelblue")
axes[1].axvline(x=optimal_thr, color="red",    linestyle="--", alpha=0.6)
axes[1].axvline(x=0.09,        color="orange", linestyle="--", alpha=0.6)
axes[1].set_xlabel("Threshold"); axes[1].set_ylabel("Rate")
axes[1].set_title("Validation: Sensitivity / Specificity Trade-off")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Test-set evaluation at fixed thresholds (chosen without using test labels).
metrics_05  = evaluate_at_threshold(y_true_best, y_prob_best, threshold=0.50)
metrics_009 = evaluate_at_threshold(y_true_best, y_prob_best, threshold=0.09)
metrics_opt = evaluate_at_threshold(y_true_best, y_prob_best, threshold=optimal_thr)

summary_df = pd.DataFrame([metrics_05, metrics_009, metrics_opt])
summary_df.insert(0, "Strategy", [
    "Tuned MLP (Default 0.5) - Test",
    "Tuned MLP (Theoretical 0.09) - Test",
    f"Tuned MLP (Validation-optimal {optimal_thr:.2f}) - Test",
])
print("Test evaluation with fixed thresholds")
summary_df

On the test set, the validation-optimal threshold achieves the lowest expected cost, improving on both the default threshold and the theoretical threshold. As expected, this operating point yields a more balanced trade-off: it preserves high sensitivity (0.922) while substantially improving specificity relative to 0.09. 

In [ ]:
# Test confusion matrices: default vs validation-optimal threshold.
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

for ax, thr, title in zip(
    axes,
    [0.5, optimal_thr],
    ["Test - Default threshold (0.50)", f"Test - Validation-optimal ({optimal_thr:.2f})"],
):
    y_pred = (y_prob_best >= thr).astype(int)
    cm = confusion_matrix(y_true_best, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No churn", "Churn"],
                yticklabels=["No churn", "Churn"])
    ax.set_title(title)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")

plt.tight_layout()
plt.show()

The confusion matrices confirm the pattern, showing that the threshold of 0.25 reduces the number of missed churners compared with the default rule, but without producing as many false alarms as the more aggressive theoretical threshold.

## 8. SHAP Interpretability

Although the tuned MLP provides flexible non-linear predictions, its internal parameters are not directly interpretable. To better understand which variables drive the model’s output, we use SHAP (SHapley Additive exPlanations).

SHAP assigns each feature an additive contribution relative to a background reference distribution, making it possible to analyse both global importance and individual predictions. In this way, SHAP helps assess whether the neural network has learned patterns that are consistent with the earlier statistical analysis.

For the neural network, we use `shap.DeepExplainer`, which is designed for deep learning models and provides an efficient approximation of feature contributions.

We present:
1. Global feature importance based on mean absolute SHAP values.
2. A beeswarm plot to show the distribution and direction of feature effects.
3. Local explanations for selected individual predictions.

In [ ]:
# ── SHAP setup ───────────────────────────────────────────────────────────────
# We need raw numpy arrays (already standardised) for the SHAP explainer.
# X_test_np comes from build_churn_loaders (already scaled).

# Retrieve standardised training data for the background distribution.
# We extract it from the train DataLoader tensors.
X_train_tensors = []
for x_batch, _ in train_loader:
    X_train_tensors.append(x_batch)
X_train_all = torch.cat(X_train_tensors, dim=0)   # shape (n_train, n_features)

# Background: a random subset of 200 training samples.
# This represents the "average" prediction against which SHAP computes contributions.
background_idx = np.random.choice(len(X_train_all), size=200, replace=False)
background = X_train_all[background_idx].to(device)

# DeepExplainer: uses backpropagation through the network to estimate SHAP values.
# It is specifically designed for neural networks (faster than KernelExplainer).
best_model_tuned.eval()
explainer = shap.DeepExplainer(best_model_tuned, background)

# Compute SHAP values for the test set (or a subset for speed).
n_explain = min(300, len(X_test_np))
X_explain = torch.tensor(X_test_np[:n_explain], dtype=torch.float32).to(device)

shap_values = explainer.shap_values(X_explain)   # shape (n_explain, n_features)
# DeepExplainer returns a list for multi-output models; for binary (1 output) it's an array.
if isinstance(shap_values, list):
    shap_values = shap_values[0]

print(f"SHAP values shape: {shap_values.shape}")
print(f"Feature names: {feature_names}")

In [ ]:
# ── 8.1 Global feature importance (mean |SHAP|) ───────────────────────────────
# The bar plot shows which features most consistently affect predictions
# across the entire test set, regardless of direction.

# SHAP can return extra singleton output dimensions depending on explainer/model.
shap_arr = np.array(shap_values)

# Common binary-shape cases: (n, p, 1) or (n, 1, p) -> collapse to (n, p)
if shap_arr.ndim == 3 and shap_arr.shape[-1] == 1:
    shap_arr = shap_arr[..., 0]
elif shap_arr.ndim == 3 and shap_arr.shape[1] == 1:
    shap_arr = shap_arr[:, 0, :]

mean_abs_shap = np.abs(shap_arr).mean(axis=0)
mean_abs_shap = np.ravel(np.squeeze(mean_abs_shap))

if mean_abs_shap.shape[0] != len(feature_names):
    raise ValueError(
        f"SHAP/features mismatch: mean_abs_shap has {mean_abs_shap.shape[0]} values "
        f"but feature_names has {len(feature_names)} entries."
    )

importance_df = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": mean_abs_shap,
}).sort_values("mean_abs_shap", ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(
    data=importance_df,
    x="mean_abs_shap",
    y="feature",
    hue="feature",
    palette="viridis_r",
    dodge=False,
    legend=False,
)
plt.xlabel("Mean |SHAP value|")
plt.title("Global Feature Importance (mean |SHAP|) - Tuned MLP")
plt.tight_layout()
plt.show()

print(importance_df.to_string(index=False))

The global SHAP analysis indicates that the tuned MLP is driven mainly by number of products and age, with additional contributions from active membership, Germany, gender, and balance. These variables are broadly aligned with the main churn-related predictors identified in Midterm 1, which supports the credibility of the neural network’s behaviour. In particular, the prominence of numofproducts suggests that the MLP may be capturing non-linear effects or interactions involving product holding that were less visible in the earlier probabilistic models.

In [ ]:
# ── 8.2 Beeswarm plot ─────────────────────────────────────────────────────────
# Shows the distribution of SHAP values per feature.
# Red = high feature value, blue = low feature value.
# SHAP > 0 pushes the prediction towards churn; SHAP < 0 pushes away from churn.

# Ensure SHAP values are 2D: (n_samples, n_features), required by beeswarm.
shap_beeswarm_values = np.array(shap_values)
if shap_beeswarm_values.ndim == 3 and shap_beeswarm_values.shape[-1] == 1:
    shap_beeswarm_values = shap_beeswarm_values[..., 0]
elif shap_beeswarm_values.ndim == 3 and shap_beeswarm_values.shape[1] == 1:
    shap_beeswarm_values = shap_beeswarm_values[:, 0, :]

shap_beeswarm_values = np.squeeze(shap_beeswarm_values)
if shap_beeswarm_values.ndim != 2:
    raise ValueError(
        f"Beeswarm expects 2D SHAP values (n_samples, n_features), got shape {shap_beeswarm_values.shape}."
    )

shap_explanation = shap.Explanation(
    values=shap_beeswarm_values,
    data=X_test_np[:n_explain],
    feature_names=feature_names,
)

plt.figure()
shap.plots.beeswarm(shap_explanation, max_display=15, show=False)
plt.title("SHAP Beeswarm Plot - Tuned MLP")
plt.tight_layout()
plt.show()

The beeswarm plot adds directional information to the global SHAP ranking. Higher age is generally associated with higher churn risk, while being an active member or a male tends to reduce it. The dummy for Germany also pushes predictions towards churn.  The pattern for number of products is especially interesting, most observations with lower values tend to push the prediction away from churn, whereas some higher values generate a strong positive contribution, suggesting a non-linear effect. Overall, the results remain broadly consistent with Midterm 1, while also indicating some non-linear structure learned by the network.

In [ ]:
# ── 8.3 Local explanations — Waterfall plots ──────────────────────────────────
# We examine two individual customers: one true churner and one loyal customer.
# The waterfall plot shows how each feature moves the prediction from the
# base rate (E[f(x)]) to the final predicted probability.

# Indices of a true churner and a true non-churner in the test set
churner_idx     = np.where(y_test_np[:n_explain] == 1)[0][0]
non_churner_idx = np.where(y_test_np[:n_explain] == 0)[0][0]

for idx, label in [(churner_idx, "True Churner"), (non_churner_idx, "True Non-Churner")]:
    plt.figure()
    # Build a local explanation with an explicit base value (required by waterfall)
    exp_val = explainer.expected_value
    if isinstance(exp_val, (list, np.ndarray)):
        exp_val = np.array(exp_val).squeeze()
    if torch.is_tensor(exp_val):
        exp_val = exp_val.detach().cpu().numpy().squeeze()
    exp_val = float(exp_val)

    local_explanation = shap.Explanation(
        values=np.array(shap_explanation.values[idx]).squeeze(),
        base_values=exp_val,
        data=np.array(shap_explanation.data[idx]).squeeze(),
        feature_names=feature_names,
    )

    shap.plots.waterfall(local_explanation, max_display=12, show=False)
    plt.title(f"Local SHAP Explanation — {label} (test obs. {idx})")
    plt.tight_layout()
    plt.show()
    prob = float(torch.sigmoid(
        best_model_tuned(torch.tensor(X_test_np[idx:idx+1], dtype=torch.float32).to(device))
    ).item())
    print(f"  → Predicted churn probability: {prob:.3f}  |  True label: {int(y_test_np[idx])}\n")

The local SHAP explanations illustrate how individual predictions are driven by a combination of competing feature effects. In both examples, number of products plays a major role, but its contribution interacts with other variables such as age, active membership, Germany, and gender. The churn case is particularly informative because it shows a mixed profile: some features push strongly towards churn, while others offset that signal. This highlights that the MLP does not rely on a single predictor, but forms its prediction through the joint effect of several variables. 


Note that the SHAP waterfall plots are expressed on the model-output (logit) scale Therefore, a negative final value does not necessarily imply classification as non-churn, since the final decision depends on the operating threshold; under the selected threshold of 0.25, the first case is classified as churn (true positive) and the second as non-churn (true negative).


## 9. Comparison with Midterm 1 Probabilistic Classifiers

We now place the best MLP (at the optimal threshold) alongside the four probabilistic models from Midterm 1. The primary ranking criterion is **Expected Cost** (lower is better), as it directly reflects the business objective.

In [ ]:
# Results from Midterm 1 (cost-sensitive threshold = 0.09 for all)
midterm1_results = [
    {"Model": "QDA",       "Strategy": "No Resampling (0.09)",    "AUC": 0.817, "Accuracy": 0.608, "Sensitivity": 0.894, "Specificity": 0.535, "Precision": 0.330, "F1_Score": 0.482, "Expected_Cost": 5.853},
    {"Model": "Naive Bayes","Strategy": "Oversampling (0.09)",    "AUC": 0.839, "Accuracy": 0.441, "Sensitivity": 0.961, "Specificity": 0.308, "Precision": 0.262, "F1_Score": 0.412, "Expected_Cost": 6.313},
    {"Model": "LDA",        "Strategy": "SMOTE (0.09)",           "AUC": 0.779, "Accuracy": 0.248, "Sensitivity": 0.998, "Specificity": 0.056, "Precision": 0.213, "F1_Score": 0.351, "Expected_Cost": 7.569},
    {"Model": "Logit",      "Strategy": "SMOTE (0.09)",           "AUC": 0.779, "Accuracy": 0.240, "Sensitivity": 0.998, "Specificity": 0.046, "Precision": 0.211, "F1_Score": 0.348, "Expected_Cost": 7.644},
]

# Best MLP results at optimal threshold
mlp_row = evaluate_at_threshold(y_true_best, y_prob_best, threshold=optimal_thr)
mlp_row["Model"]    = "MLP (Neural Network)"
mlp_row["Strategy"] = f"pos_weight + Validation opt. ({optimal_thr:.2f})"
mlp_row.pop("threshold")

comparison_df = pd.DataFrame(midterm1_results + [mlp_row])
comparison_df = comparison_df.sort_values("Expected_Cost")
print("=" * 80)
print("Comparative evaluation")
print("=" * 80)
comparison_df

In [ ]:
# Visual comparison: Expected Cost and AUC side by side.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ["tomato" if m == "MLP (Neural Network)" else "steelblue"
          for m in comparison_df["Model"]]

axes[0].barh(comparison_df["Model"], comparison_df["Expected_Cost"], color=colors)
axes[0].set_xlabel("Expected Cost per customer (lower is better)")
axes[0].set_title("Expected Cost Comparison")
axes[0].axvline(x=comparison_df["Expected_Cost"].min(), color="red", linestyle="--", alpha=0.5)

axes[1].barh(comparison_df["Model"], comparison_df["AUC"], color=colors)
axes[1].set_xlabel("AUC (higher is better)")
axes[1].set_title("AUC Comparison")
axes[1].set_xlim(0.7, 0.9)

plt.tight_layout()
plt.show()

The final comparison shows that the tuned MLP achieves the best overall performance among the models considered. It obtains both the lowest expected cost and the highest AUC, outperforming the best probabilistic competitor, QDA, on both criteria. In practical terms, the neural network preserves a high sensitivity (0.929) while maintaining a more balanced specificity than the more aggressive Midterm 1 classifiers, which makes it the strongest overall option for this churn prediction setting.

## Conclusions

This notebook extended the Midterm 1 analysis by introducing a **Multi-Layer Perceptron (MLP)** as a non-linear alternative to the probabilistic classifiers studied previously. The goal was to examine whether a neural network could capture more complex structure in the churn data while still remaining interpretable.

The results show that a relatively simple MLP is already able to learn meaningful predictive patterns from this tabular dataset. The tuning exercise also suggests that increasing architectural complexity does not provide substantial additional benefit, so the main strength of the neural network here lies in its flexibility rather than in model size.

An important contribution of this second stage is interpretability. Through SHAP, the MLP can be analysed in a way that remains connected to the earlier statistical results. The model relies mainly on variables such as **number of products**, **age**, **active membership**, **Germany**, and **balance**, while local explanations show how predictions arise from the combination of several competing feature effects.

From a practical perspective, for tabular data of this size, a neural network does not necessarily deliver a dramatic advantage over strong classical methods such as QDA. Even so, the MLP offers a richer representation of non-linearities and interactions between predictors, and this flexibility may become more valuable in larger or more complex datasets.

Overall, the neural-network approach provides a useful extension of the Midterm 1 models: it performs strongly, captures more complex structure, and remains interpretable enough to support discussion of the main drivers of customer churn.
